# CryoSwath track comparison with EOLIS

EOLIS provides CryoSat swath elevation points. This notebook compares their
elevations and positions with CryoSwath points from the same track, starting
with coverage and then examining the matched differences.

Use the CryoSwath test environment and a configured processing project with ESA
PDS credentials. If needed, set `CRYOSWATH_CONFIG` before starting the kernel.
Relevant EOLIS files are downloaded and cached automatically. Existing CryoSwath
L2 is reused; missing L2 is processed automatically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from cryoswath import eolis

%matplotlib inline

TRACK = "20240215T195335"
OUTPUT = Path("artifacts/eolis")

In [ ]:
result = eolis.compare_track(TRACK)
points = result["points"]
matched = points.loc[points.matched]

## Comparison coverage

Inspect the counts before interpreting the differences:

- `cryoswath_points`: CryoSwath points with finite heights.
- `eolis_observations`: EOLIS observations read for this track after exact duplicates
  are removed; `exact_duplicates_removed` records how many were removed.
- `selected_observations`: distinct EOLIS observations within the along-track windows.
- `matched_observations`: observations associated with an available CryoSwath sample
  and accepted by the range tolerance.
- `missing_cached_samples` and `outside_radial_tolerance`: unsuccessful candidate
  associations. An observation may be considered for several waveforms, so these
  counts do not partition the distinct selected observations.

These counts describe the EOLIS comparison, not the fraction of all CryoSwath
points covered. Only matched observations contribute to the statistics and plots.
If there are no matches, those results contain no comparison data.

In [ ]:
display(result["coverage"])

## Elevation and positional differences

Association uses satellite geometry and the nearest native waveform range sample;
elevation agreement is not a matching condition.

Signed differences are CryoSwath minus EOLIS, in metres: positive elevation means
CryoSwath is higher, along-track is forward, and across-track is right of flight.
Horizontal distance is the norm of the along/across difference.

Look for offsets from zero and the spread of the differences. These describe
agreement between products, not independent accuracy estimates. Height datum and
processing choices must be compatible; no extra quality or uncertainty filtering
is applied. The 2.5/97.5 percentiles describe the distribution, not confidence
intervals.

In [ ]:
labels = {
    "height_difference_m": "Elevation difference (m)",
    "along_difference_m": "Along-track difference (m)",
    "across_difference_m": "Across-track difference (m)",
    "horizontal_distance_m": "Horizontal distance (m)",
}
display(result["statistics"].loc[list(labels)].rename(index=labels))

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, column in zip(
    axes, ["height_difference_m", "along_difference_m", "across_difference_m"]
):
    ax.hist(matched[column].dropna(), bins=60)
    ax.set_xlabel(labels[column])
    ax.set_ylabel("Observations")
    ax.axvline(0, color="black", linestyle="--")
fig.savefig(OUTPUT / "track-comparison-differences.png", dpi=160)
plt.show()

## Spatial pattern

Use EOLIS locations to see where elevation differences change along the track.
Colors show CryoSwath minus EOLIS elevation in metres.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
if not matched.empty:
    matched.plot(
        column="height_difference_m",
        ax=ax,
        markersize=3,
        cmap="RdBu_r",
        legend=True,
        legend_kwds={"label": "Elevation difference (m)"},
    )
else:
    ax.text(0.5, 0.5, "No matched observations", ha="center", transform=ax.transAxes)
ax.set_title("Elevation differences at EOLIS locations")
ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
fig.savefig(OUTPUT / "track-comparison-map.png", dpi=160)
plt.show()

## Export results

Write CSV tables and figures to `OUTPUT`, relative to the kernel's working directory.
The point table includes unmatched candidates and their `status`; geometry is
omitted from the CSV, while other point fields are retained. The exported statistics
also include range diagnostics defined in the EOLIS comparison guide.

In [ ]:
points.drop(columns="geometry").to_csv(
    OUTPUT / "track-comparison-points.csv", index=False
)
result["coverage"].to_csv(OUTPUT / "track-comparison-coverage.csv")
result["statistics"].to_csv(OUTPUT / "track-comparison-statistics.csv")